<a href="https://colab.research.google.com/github/kovexison/Practica-AI/blob/AI-56-M3.3-Train-the-model-and-plot-training-curves/DL_CNN_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Introduction to CNNs for Image Classification

Until now, you've learned how to apply machine learning to structured datasets (like predicting prices or classifying data based on features). But how can we teach a machine to recognize images, like telling the difference between a **headlight**, a **bumper**, or a **wheel**?

That’s where **Convolutional Neural Networks (CNNs)** come in. CNNs are the **go-to architecture** in deep learning when it comes to visual understanding.

---

## 🧩 The Challenge with Image Data

Images are not like tabular data. A single color image of size 128×128 has **49,152 features** (128 × 128 × 3), which:
- Makes them **high-dimensional**
- Contains **spatial relationships** (e.g. a tire is round, and a headlight has glassy edges)
- Makes flattening them (as we did in ML) **lose important context**

Traditional ML methods treat all pixels independently. But CNNs understand **space** — they "look" at nearby pixels together, like how our eyes and brains work.

---

## 👁️ What is a CNN?

CNNs (Convolutional Neural Networks) are **neural networks designed specifically for images**. They learn patterns from raw pixel data through **filters** that scan the image.


![How a CNN Works](./images_gifs/cnnimage.png)


### Think of CNNs as layers of virtual eyes:
1. **Early layers** detect **edges**, **lines**, and **color patterns**
2. **Middle layers** detect **shapes**, **textures**, or **symmetry**
3. **Later layers** detect **parts**, **objects**, or **combinations**

All this happens without us manually designing the features. CNNs learn them during training.

---

![Digits](images_gifs/cnndigits.gif)

### 💡 How a CNN Works (Layer Ideas)

**Convolution Layer**
➤ Learns small visual patterns (like edges, curves, textures)
➤ Uses filters that slide over the image

**Activation (ReLU)**
➤ Keeps only useful signals (sets negatives to 0)
➤ Adds non-linearity so it can learn complex patterns

**Pooling Layer**
➤ Shrinks the feature maps (like zooming out)
➤ Keeps the strongest features (e.g., max pooling)

**More Conv + Pool Layers**
➤ Deeper layers learn higher-level parts (like car lights, tires)
➤ Each layer builds on the previous ones

**Flatten Layer**
➤ Turns 2D feature maps into a 1D vector
➤ Prepares data for the final decision

**Fully Connected (Dense) Layers**
➤ Combine all features to predict the class
➤ Similar to layers in traditional neural networks

**Softmax Output**
➤ Outputs class probabilities
➤ The highest value is the model’s prediction

![convs](images_gifs/convolutions.gif)

### Your Challenge:

In this notebook, we’ll train a CNN to recognize different **car parts** from images. These might include parts like:
- air compressor
- battery
- brake caliper
- coil spring
- oil pan and so on

**Total of 50 car parts**


You don’t need to **tell** the model what a headlight looks like — it will **learn** the patterns by itself from labeled training images.


By the end of this notebook, you'll be able to:

✅ Load and preprocess image data  
✅ Build your own CNN using TensorFlow/Keras  
✅ Train and evaluate the model  
✅ Visualize performance with graphs and confusion matrices  
✅ Experiment with changes and improve accuracy  


## 🖼️ Let's Look at Some Examples!

Before we start coding, here are some sample images from our dataset (randomly picked from different classes):

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

dataset_path = './content/drive/MyDrive/dataset/train'

import os

print(os.getcwd)

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import matplotlib.pyplot as plt
import os
import random

from IPython.core.pylabtools import figsize
from tensorflow.keras.preprocessing import image
from PIL import Image

class_dirs = [d for d in os.listdir(dataset_path)
              if os.path.isdir(os.path.join(dataset_path, d)) and not d.startswith('.')]

sampled_classes = random.sample(class_dirs, min(9, len(class_dirs)))

fig, axs = plt.subplots(nrows=3, ncols=3, figsize=(15, 6))
for i, class_name in enumerate(sampled_classes):
    class_folder = os.path.join(dataset_path, class_name)

    valid_extensions = ['.jpg', '.jpeg', '.png']
    images_in_class = [f for f in os.listdir(class_folder)
                       if f.lower().endswith(tuple(valid_extensions)) and not f.startswith('.')]

    if not images_in_class:
        continue

    sample_image = random.choice(images_in_class)
    img_path = os.path.join(class_folder, sample_image)

    img = image.load_img(img_path, target_size=(128, 128))
    axs[i//3, i%3].imshow(img)
    axs[i//3, i%3].set_title(class_name.title(), fontsize=10)
    axs[i//3, i%3].axis("off")

plt.suptitle("Sample Images from Different Car Part Classes", fontsize=16)
plt.tight_layout()
plt.show()


NameError: name 'dataset_path' is not defined

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv('./dataset/car parts.csv')
df.head(20)

In [ ]:
label_counts = df['labels'].value_counts().sort_values(ascending=False)
label_counts

#### TODO: Research about data augmentation and what to do in case of an unbalanced dataset
##### At the end of the notebook, you can apply these techniques and train another CNN and compare results

### Your task:

#### Quick CNN Workflow for Image Classification

1. **Load & preprocess data**  
   - Resize images and normalize pixel values (e.g., scale pixels to [0,1])  
   - Prepare labels for classification  

2. **Create training and validation data generators**  
   - Use `ImageDataGenerator` or custom loaders to batch and augment data  (If you want to, or skip this part for now)

3. **Build the CNN model**  
   - Stack convolutional layers (Conv2D + activation function) and pooling layers
   - Flatten and add dense layers for final classification  
   - Make sure to use the right activation function for the last Dense layer.

4. **Compile the model**  
   - Use suitable loss function nd an optimizer like Adam  

5. **Train the model**  
   - Fit the model on training data and validate on validation data over multiple epochs  
   - Find the right number of batches/epochs for your data

6. **Evaluate and visualize results**  
   - Plot Accuracy & Loss Curves for training/validation
   - Learn about overfitting/underfitting and how to prevent these situations
   - Generate confusion matrix and classification report, based on metrics like accuracy

7. **Explore**
   - Use a pretrained available model
   - Change the architecture
   - Integrate the model into a small GUI and make it more interactive
   - Test with images from the internet, and see how well it performs


# Part 1: Visualize and Process the Data

## Chapter 1: take a look

In [ ]:
import seaborn as sns

plt.figure(figsize=(20, 8))
sns.countplot(x=df['labels'])
plt.title("Distribution of car parts")
plt.xlabel("Car Part")
plt.ylabel("Count")
plt.xticks(rotation=85)
plt.tight_layout()
plt.show()

print(f"Our set has a total of {df.shape[0]} examples distributed among {len(label_counts)} car parts and test-train-validation splits; we see that the lowest count is {min(label_counts)}, whereas the highest count is {max(label_counts)}.")
print(f"This is a difference of {(min(label_counts)*100/max(label_counts)):.4f}% or a ratio of {max(label_counts) / min(label_counts):.2f}.")

While not ideal, this is not a concern for us right now as this imbalance is considered a mild to moderate imbalance. Problems arise when the smallest class has only a few dozen examples (we have 120) or the ratio is something like 10:1 or above.
If we see that the underrepresented classes have lower scores during training or testing we can address this issue.
However, the real problem might be the fact that we have so many categories and each category represents only a small portion of the dataset. This problem can be addressed by using other techniques like transfer learning or data augmentation to all the classes.

In [ ]:
# read the images and show the distribution of their sizes
from pathlib import Path
import numpy as np
import cv2

dataset = Path('./dataset')
widths = []
heights = []
brightness_values = []
image_array = []

fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(20, 5))
for img_path in dataset.rglob("*"):
    if img_path.suffix.lower() in {".jpg", ".jpeg", ".png"}:

        # width-height measurements
        with Image.open(img_path) as img:
            widths.append(img.width)
            heights.append(img.height)

        # brightness measurements
        img = cv2.imread(str(img_path))
        if img is not None:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            brightness_values.append(np.mean(gray))

        # compute average image
        try:
            img = Image.open(str(img_path)).convert("RGB")
            arr = np.asarray(img, dtype=np.float32) / 255.0
            image_array.append(arr)
        except Exception as e:
            print(f"{img_path} is an invalid image.")

sns.histplot(widths, bins=30, kde=True, ax=axs[0])
axs[0].set_title("Distribution of Image Widths")
axs[0].set_xlabel("Width (pixels)")
axs[0].set_ylabel("Count")
axs[0].grid(alpha=0.3)

sns.histplot(heights, bins=30, kde=True, ax=axs[1])
axs[1].set_title("Distribution of Image Heights")
axs[1].set_xlabel("Height (pixels)")
axs[1].set_ylabel("Count")
axs[1].grid(alpha=0.3)

sns.histplot(brightness_values, bins=30, kde=True, ax=axs[2])
axs[2].set_title("Distribution of Image Brightness")
axs[2].set_xlabel("Brightness (0-255)")
axs[2].set_ylabel("Count")
axs[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"A random image's height: {random.choice(heights)} and a random image's width: {random.choice(widths)}")

# display the average image
images = np.stack(image_array, axis=0)
avg_image = images.mean(axis=0)
avg_image_255 = (avg_image * 255).astype(np.uint8)
plt.figure(figsize=(6, 6))
plt.imshow(avg_image_255)
plt.title("Average Image")
plt.axis("off")
plt.show()

The dataset contains images with a uniform resolution of 224x224 pixels and an aspect ratio of 1:1. This is a good starting point as we don't have to implement any pipelines that resize or crop the images,  and the model won't have to deal with noise (different resolutions) when it classifies the images. The situation is good regarding brightness as well; although it varies a bit, there is only one peak at ~200, with most of the images around 100-200 range. There are only a few dark images, which is a good point. This relatively uniform distribution of the brightness suggests that the dataset is consistent and most probably is taken from one source only.
Meanwhile, the average image is... boring? While the idea is interesting, we can't learn too much from it. We see that usually images are in the center, which is good as it is clearer for the model where the subject is. The brightness is also relatively high, as discussed in the brightness histogram

## Chapter 2: build a pipeline

We use pipelines to streamline the process of preprocessing the images, such as standardization and normalization (reduce RGB values to [0, 1]. This ensures that the images are in a consistent format before being fed into the CNN model.

In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    "./dataset/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical"
)

valid_ds = tf.keras.utils.image_dataset_from_directory(
    "./dataset/valid",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical"
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    "./dataset/test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical"
)

In [ ]:
# Normalize pixel values to [0, 1]
normalization_layer = tf.keras.layers.Rescaling(1./255)

train_ds = train_ds.map(
    lambda x, y: (normalization_layer(x), y)
)

valid_ds = valid_ds.map(
    lambda x, y: (normalization_layer(x), y)
)

test_ds = test_ds.map(
    lambda x, y: (normalization_layer(x), y)
)

So far we have loaded the dataset into batches and standardized and normalized pixels values, essentially creating a pipeline. Now it's time we test the pipeline and see if it works as expected.

In [ ]:
for images, labels in train_ds.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)

for images, labels in train_ds.take(1):
    print("\nMinimum pixel value:", tf.reduce_min(images).numpy())
    print("Maximum pixel value:", tf.reduce_max(images).numpy())

The correct range for pixel values is [0, 1], which is what we see here. Plus, we see that each batch contains 32 images of size 224x224 with 3 color channels (RGB).

# Part 2: build a simple, baseline CNN model

## Chapter 1: initialize the model

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten, Dense)

model = Sequential([
    # Conv Block 1
    Conv2D(filters=32, kernel_size=(3, 3), activation='relu', padding='same', input_shape=(224, 224, 3)),
    MaxPooling2D(pool_size=(2, 2)),

    # Conv Block 2
    Conv2D(filters=64, kernel_size=(3, 3), activation='relu', padding='same'),
    MaxPooling2D(pool_size=(2, 2)),

    # Conv Block 3
    Conv2D(filters=128, kernel_size=(3, 3), activation='relu', padding='same'),
    MaxPooling2D(pool_size=(2, 2)),

    Flatten(),
    Dense(256, activation='relu'),
    Dense(50, activation='softmax')
])

model.summary()

We have created a CNN with 3 convolution layers (each followed by a max pooling layer) and 2 dense layers. As you can see, the model is big with a size of almost 100 MB, most of which are in the first dense layer. This is typical for CNNs, as dense layers connect every neuron to every other neuron and feature from the flattened output of convolutional layers.

## Chapter 2: compile and train the model

Compiling the model involves configuring the model's loss function, optimizer and metrics. It has nothing to do with code compiling like in C.

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy', # used because we are dealing with categorical data
    metrics=['accuracy'])

In [ ]:
history = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=20)